In [ ]:
from braket.program_sets import ProgramSet
from mpqp import QCircuit
from mpqp.gates import *
from sympy import Symbol
from braket.program_sets import CircuitBinding
from braket.devices import LocalSimulator

In [ ]:
from sympy import Symbol
from braket.program_sets import CircuitBinding
from braket.devices import LocalSimulator

t = Symbol("t")
r = Symbol("r")

c1 = QCircuit([Rz(t, 0), Rx(r, 0), Rz(t, 0)])

cb = CircuitBinding(c1, input_sets={"t": (0, 1), "r": (1, 0.01)})
s1 = ProgramSet(cb, 100)
device = LocalSimulator()
result_2 = device.run(s1).result()
for r in result_2:
    for rr in r:
        print(rr.counts)

Counter({'0': 76, '1': 24})
Counter({'0': 100})


In [ ]:
from typing import Literal, overload

from mpqp.core.circuit import BindingMode, CircuitBinding as MPQPBinding
from mpqp import QCircuit, Language
from mpqp.core.instruction.measurement.expectation_value import (
    ExpectationMeasure,
    Observable,
)
from braket.program_sets import ProgramSet


@overload
def test(bind: MPQPBinding) -> ProgramSet: ...
@overload
def test(bind: MPQPBinding, programSet: Literal[True]) -> ProgramSet: ...
@overload
def test(bind: MPQPBinding, programSet: Literal[False]) -> MPQPBinding: ...
def test(bind: MPQPBinding, programSet: bool = True) -> ProgramSet | MPQPBinding:
    from braket.program_sets import ProgramSet
    from braket.circuits import Circuit

    # translate inner circuits to braket and CB's elements to Braket
    translated: list[MPQPBinding | Circuit] = []
    for c in bind.circuits:
        if isinstance(c, QCircuit):
            translated.append(c.to_other_language(Language.BRAKET))
        else:
            translation = test(c, False)

            if isinstance(translation, list):
                translated.extend(translation)
            else:
                assert isinstance(translation, MPQPBinding)
                translated.append(translation)

    # translate var
    var = []
    if bind.value:
        from copy import deepcopy

        var = deepcopy(bind.value)
        converted = []
        for variables in var:
            current = {}
            for key in variables.keys():
                val = variables[key]
                if not isinstance(val, float | int):
                    raise ValueError(
                        f"Cannot use complex parameters in Braket. Got: {val}"
                    )
                current.update({str(key): [val]})
            converted.append(current)
        var = converted

    from braket.program_sets import CircuitBinding

    obs = []
    if bind.measurements:
        for m in bind.measurements:
            assert isinstance(m, ExpectationMeasure)
            obs.extend([o.to_other_language(Language.BRAKET) for o in m.observables])

    if not programSet:  # If the circuitBinding is embedded store the translated data.
        bind._translated_circuits = translated
        bind._translated_observables = obs
        bind._translated_variables = var
        return bind

    result = []
    # i is used only if the binding mode is zip
    if len(translated) == 1 and (
        not isinstance(translated[0], MPQPBinding) or len(translated[0].circuits) == 1
    ):
        # if i == -1 then we're in the case of a single circuit in the binding.
        # Otherwise it'll iterate of the translated circuit (and bindings) and apply the values and obs accordingly
        i = -1
    else:
        i = 0
    if bind.mode == BindingMode.ZIP:
        executables = list(zip(var or [None] * len(obs), obs or [None] * len(var)))
    else:
        from itertools import product

        executables = list(product(var or [None], obs or [None]))
    for values, observable in executables:
        if bind.mode == BindingMode.PRODUCT:
            for t in translated:
                if isinstance(t, MPQPBinding):
                    if t._translated_observables and observable:
                        raise ValueError(
                            "Cannot declare an observable both inside a CircuitBinding and outside"
                        )
                    if t._translated_variables and values:
                        raise ValueError(
                            "Cannot declare variables both inside a CircuitBinding and outside"
                        )
                    from itertools import product

                    inside_executables = list(
                        product(
                            t._translated_observables or [observable],
                            t._translated_variables or [values],
                        )
                    )
                    assert isinstance(t._translated_circuits, list)
                    for inside_observable, inside_val in inside_executables:
                        result.extend(
                            [
                                CircuitBinding(
                                    c,
                                    input_sets=inside_val,
                                    observables=[inside_observable],
                                )
                                for c in t._translated_circuits
                            ]
                        )

                else:
                    result.append(
                        CircuitBinding(t, input_sets=values, observables=[observable])
                    )
        else:
            if isinstance(
                translated[i], MPQPBinding
            ):  # ZIP to Binding ==> distribute level 0 to level 1
                if translated[i]._translated_observables and observable:
                    raise ValueError(
                        "Cannot declare an observable both inside a CircuitBinding and outside"
                    )
                if translated[i]._translated_variables and values:
                    raise ValueError(
                        "Cannot declare variables both inside a CircuitBinding and outside"
                    )
                from itertools import product

                inside_executables = list(
                    product(
                        translated[i]._translated_observables or [observable],
                        translated[i]._translated_variables or [values],
                    )
                )
                if i == -1:
                    for inside_observable, inside_val in inside_executables:
                        result.append(
                            CircuitBinding(
                                translated[i]._translated_circuits[0],
                                input_sets=inside_val,
                                observables=[inside_observable],
                            )
                        )
                else:
                    for inside_observable, inside_val in inside_executables:
                        assert inside_observable
                        result.extend(
                            [
                                CircuitBinding(
                                    c,
                                    input_sets=inside_val,
                                    observables=[inside_observable],
                                )
                                for c in translated[i]._translated_circuits
                            ]
                        )
            else:

                if i == -1:
                    result.append(
                        CircuitBinding(
                            translated[0], input_sets=values, observables=[observable]
                        )
                    )
                else:
                    result.append(
                        CircuitBinding(
                            translated[i],
                            input_sets=values,
                            observables=[observable],
                        )
                    )
                    i += 1

    from braket.program_sets import ProgramSet

    return ProgramSet(result, bind.shots)


from braket.program_sets import ProgramSet
from mpqp import QCircuit
from mpqp.gates import *
from sympy import Symbol
from mpqp.core.instruction.measurement.pauli_string import pX, pZ

t = Symbol("t")
r = Symbol("r")
c1 = QCircuit([Ry(t, 0), Rx(r, 0)])
c2 = QCircuit([Rz(t, 0), Rz(r, 0)])
o = ExpectationMeasure([Observable(pX), Observable(pZ)])

2


In [ ]:
t_should_error_1 = MPQPBinding(
    [
        MPQPBinding(
            c1, values=[{"t": 1.0, "r": 0.0}, {"t": 0.0, "r": 1.0}], measurements=o
        )
    ],
    mode=BindingMode.ZIP,
)

tzip_error = MPQPBinding(
    c1,
    values=[{"t": 1.0, "r": 0.0}, {"t": 0.0, "r": 1.0}, {"t": 0.0, "r": 1.0}],
    measurements=o,
    mode=BindingMode.ZIP,
)

In [43]:
tz1_4 = MPQPBinding(
    [MPQPBinding(c1, values=[{"t": 1.0, "r": 0.0}, {"t": 0.0, "r": 1.0}])],
    measurements=o,
    mode=BindingMode.ZIP,
)
tz1_2 = MPQPBinding(
    c1,
    values=[{"t": 1.0, "r": 0.0}, {"t": 0.0, "r": 1.0}],
    measurements=o,
    mode=BindingMode.ZIP,
)

t_8 = MPQPBinding(
    [c1, c2],
    values=[{"t": 1.0, "r": 0.0}, {"t": 0.0, "r": 1.0}],
    measurements=o,
    mode=BindingMode.PRODUCT,
)

t_mix_3 = MPQPBinding(
    [c2, MPQPBinding(c1, values=[{"t": 1.0, "r": 0.0}, {"t": 0.0, "r": 1.0}])],
    measurements=o,
    mode=BindingMode.ZIP,
)

t_mix_6 = MPQPBinding(
    [
        c2,
        MPQPBinding(
            c1,
            values=[{"t": 1.0, "r": 0.0}, {"t": 0.0, "r": 1.0}],
            mode=BindingMode.ZIP,
        ),
    ],
    measurements=o,
    mode=BindingMode.PRODUCT,
)

t_mix_8 = MPQPBinding(
    [c2, MPQPBinding(c1)],
    values=[{"t": 1.0, "r": 0.0}, {"t": 0.0, "r": 1.0}],
    measurements=o,
    mode=BindingMode.PRODUCT,
)

a = [tz1_4, tz1_2, t_mix_6, t_8, t_mix_3, t_mix_8]
b = ["z1 = 4", "z1 = 2", "mix_6", "8", "mix_3", "mix_8"]

for a_, b_ in zip(a, b):
    bruv = test(a_)
    print(b_)
    print(str(len(bruv)) + "\n")

z1 = 4
4

z1 = 2
2

mix_6
6

8
8

mix_3
3

mix_8
8

